# Vaccine Parameter Derivation

This notebook documents and verifies the derivation of Arrhenius pre-exponential factors (A values)
from WHO-published shelf-life specifications.

## Method

For each vaccine, given:
- `Ea` — activation energy (J/mol)
- `t_shelf` — shelf life in hours
- `P_min` — minimum acceptable potency at end of shelf life (fraction)
- `T_ref` — reference storage temperature (K)

The rate constant at reference temperature is:
```
k_ref = -ln(P_min) / t_shelf
```

And A is derived from the Arrhenius equation:
```
A = k_ref / exp(-Ea / (R * T_ref))
```


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from core.arrhenius import derive_A_from_shelf_life, compute_potency
from core.vaccine_params import VACCINE_DB

R_GAS = 8.314  # J/(mol·K)

## 1. Tabulate Derived Parameters

In [ ]:
rows = []
for key, vp in VACCINE_DB.items():
    A_derived = derive_A_from_shelf_life(
        Ea=vp.Ea_mean,
        shelf_life_hours=vp.shelf_life_hours,
        min_potency=vp.min_potency_threshold,
        T_ref_K=vp.ref_temp_K
    )
    
    # Verify round-trip: potency at shelf life end ≈ min_potency_threshold
    ts = [0, vp.shelf_life_hours * 3600]
    T_ref_C = vp.ref_temp_K - 273.15
    temps = [T_ref_C, T_ref_C]
    potency_at_shelf_end = compute_potency(ts, temps, vp.Ea_mean, A_derived)
    
    rows.append({
        'Vaccine': key,
        'Ea (kJ/mol)': vp.Ea_mean / 1000,
        'Shelf Life (months)': vp.shelf_life_hours / 720,
        'Min Potency': vp.min_potency_threshold,
        'T_ref (°C)': vp.ref_temp_K - 273.15,
        'A (derived)': f'{A_derived:.3e}',
        'A (stored)': f'{vp.A:.3e}',
        'Potency at shelf end': f'{potency_at_shelf_end*100:.2f}%',
        'Match': '✓' if abs(potency_at_shelf_end - vp.min_potency_threshold) < 0.005 else '✗'
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 2. Sensitivity Analysis: Effect of Ea Uncertainty

How does uncertainty in Ea affect the estimated potency?

In [ ]:
import matplotlib.pyplot as plt

vp = VACCINE_DB['DPT']

# 1-week storage at 4°C
n_hours = 168
timestamps = list(np.arange(n_hours + 1) * 3600.0)
temperatures = [4.0] * (n_hours + 1)

# Vary Ea by ±3 sigma
Ea_values = np.linspace(vp.Ea_mean - 3*vp.Ea_std, vp.Ea_mean + 3*vp.Ea_std, 50)
potencies = [compute_potency(timestamps, temperatures, Ea, vp.A) for Ea in Ea_values]

fig, ax = plt.subplots()
ax.plot((Ea_values - vp.Ea_mean) / vp.Ea_std, [p*100 for p in potencies], 'b-', linewidth=2)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Mean Ea')
ax.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% threshold')
ax.set_xlabel('Ea deviation (sigma)')
ax.set_ylabel('Potency (%)')
ax.set_title('DPT Potency Sensitivity to Ea Uncertainty\n(1 week at 4°C)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Ea uncertainty range: {vp.Ea_mean-vp.Ea_std:.0f}–{vp.Ea_mean+vp.Ea_std:.0f} J/mol')
print(f'Corresponding potency range: {min(potencies)*100:.2f}%–{max(potencies)*100:.2f}%')

## 3. MKT vs Full Trajectory Comparison

WHO MKT gives a single equivalent temperature; ColdGuard integrates the full trajectory.

In [ ]:
from core.arrhenius import compute_mkt, mkt_potency_estimate

# Asymmetric excursion: brief high temperature
n = 169
rng = np.random.default_rng(99)
temps = 4.0 + rng.normal(0, 0.3, n)
temps[60:64] = 35.0  # 4-hour excursion to 35°C

timestamps = list(np.arange(n) * 3600.0)
vp = VACCINE_DB['OPV']

mkt = compute_mkt(temps.tolist(), vp.Ea_mean)
mkt_pot = mkt_potency_estimate(temps.tolist(), timestamps, vp.Ea_mean, vp.A)
full_pot = compute_potency(timestamps, temps.tolist(), vp.Ea_mean, vp.A)

print(f'OPV — 4hr excursion to 35°C scenario:')
print(f'  MKT:                  {mkt:.2f}°C')
print(f'  MKT-based potency:    {mkt_pot*100:.2f}%')
print(f'  Full trajectory:      {full_pot*100:.2f}%')
print(f'  Difference:           {abs(mkt_pot - full_pot)*100:.3f}%')